# LeetCode 3289 — The Two Sneaky Numbers of Digitville bằng Python

Link đề: https://leetcode.com/problems/the-two-sneaky-numbers-of-digitville/ · Độ khó: **Easy** · Chủ đề: Array, Hash Table, Math

Skeleton LeetCode:

```python
class Solution:
    def getSneakyNumbers(self, nums: List[int]) -> List[int]:
        ...
```

Ghi chú: mọi đáp án trong notebook đã được kiểm chứng bằng code (kể cả đối chiếu hàng nghìn mảng ngẫu nhiên đúng ràng buộc bằng hàm kiểm tra tính hợp lệ độc lập). Ba ví dụ chính lấy từ đề; ví dụ nào tự dựng thêm có ghi rõ "tự dựng" — nếu muốn khớp từng chữ thì nên đối chiếu lại đề gốc.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: mảng `nums` đáng lẽ chứa các số `0..n-1`, mỗi số **đúng một lần**, nhưng có **hai số** "lén" xuất hiện **thêm một lần nữa** nên mảng dài `n + 2`; hãy tìm hai số bị lặp đó (thứ tự bất kỳ).
2. Nắm các công cụ Python: **`set`** (tập hợp, kiểm tra `x in seen` rất nhanh), **`dict`/`Counter`** đếm tần suất, **mảng đếm** `cnt[x] += 1`, **`sorted`** (so với `.sort()` sửa tại chỗ), **`list.count`**, `math.isqrt` và công thức **tổng `0..n-1`**.
3. Đi từ cách **so từng cặp** → **sắp xếp rồi so hai phần tử kề nhau** → **tập đã gặp `seen`** → **mảng/bộ đếm tần suất** → **toán học: tổng và tổng bình phương** (không cần cấu trúc dữ liệu phụ).
4. Hiểu mấu chốt: "tăng số lần xuất hiện lên `2`" là **dấu hiệu** của số bị lặp; ta có thể **ghi nhớ những gì đã thấy** (`set`/mảng đếm) và phát hiện lần xuất hiện thứ hai ngay khi gặp.
5. Tránh các lỗi kinh điển: trả cả số xuất hiện `>= 1` lần (nhầm ngưỡng), thêm số lặp **hai lần** vào kết quả, `IndexError` khi so phần tử kề nhau, `return` quá sớm (chỉ tìm được một số), và dùng `/` thay cho `//` khiến ra số thực.

## 2. Đề bài & ví dụ

Ở thị trấn Digitville có một danh sách `nums` gồm các số nguyên từ `0` đến `n - 1`. Mỗi số lẽ ra chỉ xuất hiện **đúng một lần**, nhưng có **hai** số "tinh nghịch" đã chen vào **thêm một lần nữa**, làm danh sách dài hơn bình thường.

Là thám tử của thị trấn, hãy tìm hai số đó. Trả về mảng gồm **hai** số (theo thứ tự **bất kỳ**).

**Ví dụ 1**

- Input: `nums = [0,1,1,0]`
- Output: `[0,1]`
- Giải thích: số `0` và `1` mỗi số xuất hiện hai lần.

**Ví dụ 2**

- Input: `nums = [0,3,2,1,3,2]`
- Output: `[2,3]`
- Giải thích: số `2` và `3` mỗi số xuất hiện hai lần.

**Ví dụ 3**

- Input: `nums = [7,1,5,4,3,4,6,0,9,5,8,2]`
- Output: `[4,5]`
- Giải thích: số `4` và `5` mỗi số xuất hiện hai lần.

**Ràng buộc:** `2 <= n <= 100`; `nums.length == n + 2`; `0 <= nums[i] < n`; đầu vào được tạo sao cho `nums` chứa **đúng hai** phần tử bị lặp (mỗi phần tử đó xuất hiện đúng hai lần, các số còn lại đúng một lần).

**Chú ý:** `n` **không** phải `len(nums)`; ta có `n = len(nums) - 2`.

In [1]:
import math
import random
from collections import Counter
from typing import List

# (nums, kết quả kỳ vọng — thứ tự tùy ý) — ba ví dụ lấy từ đề
examples = [
    ([0, 1, 1, 0], [0, 1]),
    ([0, 3, 2, 1, 3, 2], [2, 3]),
    ([7, 1, 5, 4, 3, 4, 6, 0, 9, 5, 8, 2], [4, 5]),
]
for nums, expected in examples:
    print(f"nums={nums} (n = {len(nums) - 2}) -> kỳ vọng {expected}")

nums=[0, 1, 1, 0] (n = 2) -> kỳ vọng [0, 1]
nums=[0, 3, 2, 1, 3, 2] (n = 4) -> kỳ vọng [2, 3]
nums=[7, 1, 5, 4, 3, 4, 6, 0, 9, 5, 8, 2] (n = 10) -> kỳ vọng [4, 5]


## 3. Kiến thức nền

### 3.1. Tập hợp `set`: "đã thấy chưa?" trong `O(1)`

`set` là tập các phần tử **không trùng**. `s.add(x)` thêm `x`; `x in s` kiểm tra có mặt hay không (nhanh, trung bình `O(1)`). Thêm lại phần tử đã có thì không thay đổi gì. Đây là công cụ chính để phát hiện "lần xuất hiện thứ hai".

In [2]:
seen = set()
for x in [4, 7, 4, 9]:
    print(f"x = {x}: đã gặp chưa? {x in seen}")
    seen.add(x)
print(seen, len(seen))        # {9, 4, 7} 3  -- thứ tự trong set không đảm bảo

x = 4: đã gặp chưa? False
x = 7: đã gặp chưa? False
x = 4: đã gặp chưa? True
x = 9: đã gặp chưa? False
{9, 4, 7} 3


### 3.2. Đếm tần suất: `dict` và `Counter`

Muốn biết mỗi số xuất hiện bao nhiêu lần, ta dùng **từ điển** (`dict`) ánh xạ `số -> số lần`. `collections.Counter` làm việc đó sẵn: `Counter([1, 1, 2])` cho `{1: 2, 2: 1}`. `.items()` duyệt các cặp `(số, số lần)`.

In [3]:
cnt = Counter([0, 3, 2, 1, 3, 2])
print(cnt)                                   # Counter({3: 2, 2: 2, 0: 1, 1: 1})
print(cnt[3], cnt[0], cnt[99])               # 2 1 0  -- khóa vắng mặt cho 0, không lỗi
print([x for x, v in cnt.items() if v == 2])  # [3, 2]

# tự làm bằng dict
freq = {}
for x in [0, 3, 2, 1, 3, 2]:
    freq[x] = freq.get(x, 0) + 1
print(freq)

Counter({3: 2, 2: 2, 0: 1, 1: 1})
2 1 0
[3, 2]
{0: 1, 3: 2, 2: 2, 1: 1}


### 3.3. Mảng đếm (counting array) khi giá trị nằm trong `0..n-1`

Vì `0 <= nums[i] < n`, ta có thể dùng **mảng** `cnt = [0] * n` thay cho `dict`: `cnt[x] += 1` ghi nhận lần xuất hiện của `x`. Chỉ số **chính là** giá trị. Cách này nhanh và gọn, dùng được vì giá trị nhỏ và liên tục.

In [4]:
n = 4
cnt = [0] * n
for x in [0, 3, 2, 1, 3, 2]:
    cnt[x] += 1
print(cnt)                      # [1, 1, 2, 2]  -> số 2 và 3 xuất hiện 2 lần
print([v for v in range(n) if cnt[v] == 2])    # [2, 3]

[1, 1, 2, 2]
[2, 3]


### 3.4. `sorted` và `.sort()`; so sánh phần tử kề nhau

`sorted(a)` trả về danh sách **mới** đã sắp xếp, giữ nguyên `a`. `a.sort()` sắp xếp **ngay trên `a`** (sửa dữ liệu gốc) và trả `None`. Sau khi sắp xếp, các số **bằng nhau nằm cạnh nhau**, nên chỉ cần so `a[i]` với `a[i + 1]`. Vì có `a[i + 1]`, vòng lặp phải dừng ở `len(a) - 2` (`range(len(a) - 1)`).

In [5]:
a = [0, 3, 2, 1, 3, 2]
b = sorted(a)
print(a, b)                        # a không đổi
for i in range(len(b) - 1):        # i + 1 <= len(b) - 1 : không IndexError
    if b[i] == b[i + 1]:
        print(f"b[{i}] == b[{i + 1}] == {b[i]}")
print(a.sort(), a)                 # None [0, 1, 2, 2, 3, 3]  -- a bị sửa tại chỗ

[0, 3, 2, 1, 3, 2] [0, 1, 2, 2, 3, 3]
b[2] == b[3] == 2
b[4] == b[5] == 3
None [0, 1, 2, 2, 3, 3]


### 3.5. `list.count` và vì sao nó chậm

`a.count(x)` đếm số lần `x` xuất hiện bằng cách **quét cả danh sách** — `O(n)`. Dùng trong vòng lặp qua mọi phần tử sẽ thành `O(n²)`. Dễ viết nhưng chậm hơn `set`/`Counter`.

In [6]:
a = [7, 1, 5, 4, 3, 4]
print(a.count(4), a.count(7), a.count(100))   # 2 1 0

2 1 0


### 3.6. Một chút toán: tổng, tổng bình phương và `math.isqrt`

- Tổng `0 + 1 + ... + (n-1) = n(n-1)/2`; tổng bình phương `0² + ... + (n-1)² = (n-1)n(2n-1)/6`. Hai công thức dùng `//` vì kết quả luôn là số nguyên.
- `math.isqrt(m)` là **căn bậc hai nguyên** (làm tròn xuống) của số nguyên không âm `m`, chính xác tuyệt đối với số nguyên (khác `m ** 0.5` có thể sai số thực).
- Nếu `a`, `b` là hai số cần tìm thì biết `a + b` và `a² + b²` là đủ để tìm lại cả hai (xem Mục 7, Cách 5).

In [7]:
n = 10
print(sum(range(n)), n * (n - 1) // 2)                             # 45 45
print(sum(k * k for k in range(n)), (n - 1) * n * (2 * n - 1) // 6)  # 285 285
print(math.isqrt(16), math.isqrt(17), math.isqrt(24), math.isqrt(25))  # 4 4 4 5
print(7 / 2, 7 // 2)       # 3.5 3  -- dùng // để giữ số nguyên

45 45
285 285
4 4 4 5
3.5 3


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Nếu mọi số `0..n-1` chỉ xuất hiện một lần, mảng sẽ có `n` phần tử. Mảng thật có `n + 2` phần tử, **thừa đúng hai phần tử**, mà hai phần tử thừa ấy chính là hai số bị lặp. Vậy: **số bị lặp = số xuất hiện lần thứ hai**.

Tính tay ví dụ 2: `nums = [0, 3, 2, 1, 3, 2]` (vậy `n = 4`). Đi từ trái sang phải và ghi lại những số đã thấy:

| bước | đọc | đã thấy trước đó? | tập đã thấy sau bước |
|---|---|---|---|
| 1 | `0` | chưa | `{0}` |
| 2 | `3` | chưa | `{0, 3}` |
| 3 | `2` | chưa | `{0, 2, 3}` |
| 4 | `1` | chưa | `{0, 1, 2, 3}` |
| 5 | `3` | **rồi** ⇒ số lặp | không đổi |
| 6 | `2` | **rồi** ⇒ số lặp | không đổi |

Hai số lặp là `3` và `2` (theo thứ tự phát hiện).

In [8]:
nums = [0, 3, 2, 1, 3, 2]
seen = set()
for x in nums:
    print(f"đọc {x}: đã thấy? {x in seen}")
    seen.add(x)

đọc 0: đã thấy? False
đọc 3: đã thấy? False
đọc 2: đã thấy? False
đọc 1: đã thấy? False
đọc 3: đã thấy? True
đọc 2: đã thấy? True


## 5. Cách 1 — Brute Force / baseline: so từng cặp phần tử

**Ý tưởng:** nếu `nums[i] == nums[j]` với `i < j` thì giá trị đó bị lặp. Thử **mọi cặp** `(i, j)` với `i < j` bằng hai vòng `for` lồng nhau. Vì mỗi số lặp xuất hiện **đúng hai lần**, nó tạo ra đúng **một** cặp trùng nên không bị thêm hai lần.

In [9]:
def sneaky_bruteforce(nums: List[int]) -> List[int]:
    ans = []
    for i in range(len(nums)):
        for j in range(i + 1, len(nums)):       # j > i để mỗi cặp chỉ xét một lần
            if nums[i] == nums[j]:
                ans.append(nums[i])
    return ans


for nums, expected in examples:
    print(sneaky_bruteforce(nums), "kỳ vọng", expected)

[0, 1] kỳ vọng [0, 1]
[3, 2] kỳ vọng [2, 3]
[5, 4] kỳ vọng [4, 5]


**Độ phức tạp (Cách 1).** Có `(n+2)(n+1)/2` cặp ⇒ **Time `O(n²)`**; chỉ dùng thêm danh sách kết quả hai phần tử ⇒ **Space `O(1)`**. Với `n <= 100` chỉ khoảng `5 000` cặp, chạy ngay. **Nút thắt:** mỗi phần tử bị so với **mọi** phần tử phía sau trong khi ta chỉ cần biết "**đã gặp trước đó chưa**" — câu hỏi mà `set` trả lời trong `O(1)`.

## 6. Ý tưởng thuật toán / bước đột phá

**Hỏi "đã thấy chưa?" thay vì so từng cặp.** Ta duyệt `nums` một lượt, giữ tập `seen` các số đã gặp:

- gặp `x` **chưa** có trong `seen` ⇒ thêm `x` vào `seen`;
- gặp `x` **đã** có trong `seen` ⇒ đây là **lần xuất hiện thứ hai** ⇒ `x` là số bị lặp, ghi vào `ans`.

Mỗi số lặp xuất hiện đúng hai lần nên đúng một lần bị bắt ở lần thứ hai ⇒ `ans` gồm đúng hai số, không trùng. Mỗi bước chỉ cần `x in seen` và `seen.add(x)` — đều `O(1)` ⇒ cả bài `O(n)`.

In [10]:
nums = [7, 1, 5, 4, 3, 4, 6, 0, 9, 5, 8, 2]
seen = set()
ans = []
for x in nums:
    if x in seen:
        ans.append(x)       # lần xuất hiện thứ hai
    else:
        seen.add(x)
print(ans)                  # [4, 5]

[4, 5]


## 7. Các cách tiếp theo

### Cách 2 — Sắp xếp rồi so hai phần tử kề nhau

Sau khi sắp xếp, hai bản sao của số lặp **đứng cạnh nhau**. Duyệt `i` từ `0` đến `len - 2`: nếu `b[i] == b[i + 1]` thì `b[i]` là số lặp. Ta dùng `sorted` (trả bản mới) để **không làm hỏng** `nums` gốc. Vì mỗi số lặp xuất hiện đúng hai lần nên không bị bắt hai lần (ba lần liên tiếp mới tạo hai cặp trùng liên tiếp).

In [11]:
def sneaky_sort(nums: List[int]) -> List[int]:
    b = sorted(nums)                         # bản sao đã sắp xếp
    ans = []
    for i in range(len(b) - 1):              # dừng ở len - 2 vì có b[i + 1]
        if b[i] == b[i + 1]:
            ans.append(b[i])
    return ans


class Solution2:
    def getSneakyNumbers(self, nums: List[int]) -> List[int]:
        b = sorted(nums)
        return [b[i] for i in range(len(b) - 1) if b[i] == b[i + 1]]


for nums, expected in examples:
    print(sneaky_sort(nums), Solution2().getSneakyNumbers(nums), "kỳ vọng", expected)

[0, 1] [0, 1] kỳ vọng [0, 1]
[2, 3] [2, 3] kỳ vọng [2, 3]
[4, 5] [4, 5] kỳ vọng [4, 5]


**Độ phức tạp Cách 2:** **Time `O(n log n)`** (sắp xếp), **Space `O(n)`** (bản sao `sorted`; có thể `O(1)` nếu `nums.sort()` tại chỗ, nhưng sửa dữ liệu vào).

### Cách 3 — Tập đã gặp `seen` (một lượt)

Đây chính là thuật toán ở Mục 6: duyệt một lượt, `x in seen` ⇒ thêm vào `ans`, ngược lại `seen.add(x)`.

In [12]:
def sneaky_seen(nums: List[int]) -> List[int]:
    seen = set()
    ans = []
    for x in nums:
        if x in seen:
            ans.append(x)
        else:
            seen.add(x)
    return ans


class Solution3:
    def getSneakyNumbers(self, nums: List[int]) -> List[int]:
        seen = set()
        ans = []
        for x in nums:
            if x in seen:
                ans.append(x)
            seen.add(x)
        return ans


for nums, expected in examples:
    print(sneaky_seen(nums), Solution3().getSneakyNumbers(nums), "kỳ vọng", expected)

[1, 0] [1, 0] kỳ vọng [0, 1]
[3, 2] [3, 2] kỳ vọng [2, 3]
[4, 5] [4, 5] kỳ vọng [4, 5]


Trong `Solution3`, ta bỏ `else`: luôn `seen.add(x)` (thêm số đã có vào `set` không gây hại), gọn hơn một chút. Nhớ **kiểm tra `in` trước khi thêm**.

**Độ phức tạp Cách 3:** **Time `O(n)`** (mỗi phần tử một lần `in` và một lần `add`, trung bình `O(1)`), **Space `O(n)`** cho `seen`.

### Cách 4 — Mảng đếm / `Counter`

Đếm số lần xuất hiện của từng giá trị rồi lấy các giá trị có số lần bằng `2`. Có hai biến thể: **mảng đếm** `cnt = [0] * n` (chỉ số là giá trị, dùng được vì `0 <= x < n`) và **`Counter`**. Mảng đếm có ưu điểm **bắt ngay lần thứ hai** khi `cnt[x]` vừa trở thành `2`.

In [13]:
def sneaky_array(nums: List[int]) -> List[int]:
    n = len(nums) - 2                       # đề: nums.length == n + 2
    cnt = [0] * n
    ans = []
    for x in nums:
        cnt[x] += 1
        if cnt[x] == 2:                     # vừa thấy lần thứ hai
            ans.append(x)
    return ans


class Solution4:
    def getSneakyNumbers(self, nums: List[int]) -> List[int]:
        cnt = Counter(nums)
        return [x for x, v in cnt.items() if v == 2]


for nums, expected in examples:
    print(sneaky_array(nums), Solution4().getSneakyNumbers(nums), "kỳ vọng", expected)

[1, 0] [0, 1] kỳ vọng [0, 1]
[3, 2] [3, 2] kỳ vọng [2, 3]
[4, 5] [5, 4] kỳ vọng [4, 5]


**Độ phức tạp Cách 4:** **Time `O(n)`**, **Space `O(n)`** (mảng đếm kích thước `n`, hoặc `Counter`).

### Cách 5 — Toán học: tổng và tổng bình phương (không cần cấu trúc phụ)

Gọi `a`, `b` là hai số bị lặp. So với mảng "chuẩn" `0..n-1`, mảng thật **thừa đúng** `a` và `b`. Vậy:

- `S = sum(nums) - n(n-1)/2 = a + b`
- `Q = sum(x²) - (n-1)n(2n-1)/6 = a² + b²`

Từ `S` và `Q` ta suy ra `a`, `b`: ta có `(a - b)² = 2(a² + b²) - (a + b)² = 2Q - S²`. Đặt `d = |a - b| = isqrt(2Q - S²)` (đúng vì `2Q - S²` là số chính phương, `d > 0` do `a != b`) rồi `a = (S + d) / 2`, `b = (S - d) / 2` — nhớ dùng `//` để giữ số nguyên. Ví dụ `[0, 1, 1, 0]`: `n = 2`, `S = 2 - 1 = 1`, `Q = 2 - 1 = 1`, `d² = 2*1 - 1 = 1`, `d = 1`, ra `{1, 0}`.

In [14]:
def sneaky_math(nums: List[int]) -> List[int]:
    n = len(nums) - 2
    s = sum(nums) - n * (n - 1) // 2                                   # a + b
    q = sum(x * x for x in nums) - (n - 1) * n * (2 * n - 1) // 6      # a^2 + b^2
    d = math.isqrt(2 * q - s * s)                                      # |a - b|
    return [(s + d) // 2, (s - d) // 2]


class Solution5:
    def getSneakyNumbers(self, nums: List[int]) -> List[int]:
        n = len(nums) - 2
        s = sum(nums) - n * (n - 1) // 2
        q = sum(x * x for x in nums) - (n - 1) * n * (2 * n - 1) // 6
        d = math.isqrt(2 * q - s * s)
        return [(s + d) // 2, (s - d) // 2]


for nums, expected in examples:
    print(sneaky_math(nums), Solution5().getSneakyNumbers(nums), "kỳ vọng", expected)

[1, 0] [1, 0] kỳ vọng [0, 1]
[3, 2] [3, 2] kỳ vọng [2, 3]
[5, 4] [5, 4] kỳ vọng [4, 5]


**Độ phức tạp Cách 5:** **Time `O(n)`** (hai lần `sum`), **Space `O(1)`**. Đây là cách duy nhất không cần cấu trúc dữ liệu phụ; đổi lại hơi khó nghĩ, và chỉ đúng nhờ đề **bảo đảm** đúng hai số lặp, mỗi số đúng hai lần.

## 8. TRACE từng bước

Ta trace Cách 3 (`seen`) trên ví dụ 3, trace Cách 4 (mảng đếm) trên ví dụ 2 và trace Cách 5 (toán) trên ví dụ 1 và ví dụ 2.

In [15]:
def trace_seen(nums: List[int]) -> List[int]:
    print(f"nums = {nums}")
    seen = set()
    ans = []
    for step, x in enumerate(nums, start=1):
        if x in seen:
            ans.append(x)
            print(f"  bước {step}: x = {x} ĐÃ có trong seen -> số lặp, ans = {ans}")
        else:
            seen.add(x)
            print(f"  bước {step}: x = {x} chưa có -> thêm, seen = {sorted(seen)}")
    return ans


assert trace_seen([7, 1, 5, 4, 3, 4, 6, 0, 9, 5, 8, 2]) == [4, 5]

nums = [7, 1, 5, 4, 3, 4, 6, 0, 9, 5, 8, 2]
  bước 1: x = 7 chưa có -> thêm, seen = [7]
  bước 2: x = 1 chưa có -> thêm, seen = [1, 7]
  bước 3: x = 5 chưa có -> thêm, seen = [1, 5, 7]
  bước 4: x = 4 chưa có -> thêm, seen = [1, 4, 5, 7]
  bước 5: x = 3 chưa có -> thêm, seen = [1, 3, 4, 5, 7]
  bước 6: x = 4 ĐÃ có trong seen -> số lặp, ans = [4]
  bước 7: x = 6 chưa có -> thêm, seen = [1, 3, 4, 5, 6, 7]
  bước 8: x = 0 chưa có -> thêm, seen = [0, 1, 3, 4, 5, 6, 7]
  bước 9: x = 9 chưa có -> thêm, seen = [0, 1, 3, 4, 5, 6, 7, 9]
  bước 10: x = 5 ĐÃ có trong seen -> số lặp, ans = [4, 5]
  bước 11: x = 8 chưa có -> thêm, seen = [0, 1, 3, 4, 5, 6, 7, 8, 9]
  bước 12: x = 2 chưa có -> thêm, seen = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]


In [16]:
def trace_array(nums: List[int]) -> List[int]:
    n = len(nums) - 2
    cnt = [0] * n
    ans = []
    print(f"nums = {nums}, n = {n}, cnt ban đầu = {cnt}")
    for step, x in enumerate(nums, start=1):
        cnt[x] += 1
        note = "  <- lần thứ hai!" if cnt[x] == 2 else ""
        if cnt[x] == 2:
            ans.append(x)
        print(f"  bước {step}: x = {x}, cnt = {cnt}{note}")
    return ans


assert trace_array([0, 3, 2, 1, 3, 2]) == [3, 2]

nums = [0, 3, 2, 1, 3, 2], n = 4, cnt ban đầu = [0, 0, 0, 0]
  bước 1: x = 0, cnt = [1, 0, 0, 0]
  bước 2: x = 3, cnt = [1, 0, 0, 1]
  bước 3: x = 2, cnt = [1, 0, 1, 1]
  bước 4: x = 1, cnt = [1, 1, 1, 1]
  bước 5: x = 3, cnt = [1, 1, 1, 2]  <- lần thứ hai!
  bước 6: x = 2, cnt = [1, 1, 2, 2]  <- lần thứ hai!


In [17]:
def trace_math(nums: List[int]) -> List[int]:
    n = len(nums) - 2
    base_sum = n * (n - 1) // 2
    base_sq = (n - 1) * n * (2 * n - 1) // 6
    s = sum(nums) - base_sum
    q = sum(x * x for x in nums) - base_sq
    disc = 2 * q - s * s
    d = math.isqrt(disc)
    a, b = (s + d) // 2, (s - d) // 2
    print(f"nums = {nums}, n = {n}")
    print(f"  sum(nums) = {sum(nums)}, tổng chuẩn = {base_sum} -> a + b = {s}")
    print(f"  tổng bình phương = {sum(x * x for x in nums)}, chuẩn = {base_sq} -> a^2 + b^2 = {q}")
    print(f"  (a - b)^2 = 2*{q} - {s}^2 = {disc}, |a - b| = {d}")
    print(f"  a = ({s} + {d}) // 2 = {a}, b = ({s} - {d}) // 2 = {b}")
    return [a, b]


assert sorted(trace_math([0, 1, 1, 0])) == [0, 1]
assert sorted(trace_math([0, 3, 2, 1, 3, 2])) == [2, 3]

nums = [0, 1, 1, 0], n = 2
  sum(nums) = 2, tổng chuẩn = 1 -> a + b = 1
  tổng bình phương = 2, chuẩn = 1 -> a^2 + b^2 = 1
  (a - b)^2 = 2*1 - 1^2 = 1, |a - b| = 1
  a = (1 + 1) // 2 = 1, b = (1 - 1) // 2 = 0
nums = [0, 3, 2, 1, 3, 2], n = 4
  sum(nums) = 11, tổng chuẩn = 6 -> a + b = 5
  tổng bình phương = 27, chuẩn = 14 -> a^2 + b^2 = 13
  (a - b)^2 = 2*13 - 5^2 = 1, |a - b| = 1
  a = (5 + 1) // 2 = 3, b = (5 - 1) // 2 = 2


Bảng trace Cách 3 với `nums = [7, 1, 5, 4, 3, 4, 6, 0, 9, 5, 8, 2]`:

| bước | `x` | `x in seen`? | `ans` |
|---|---|---|---|
| 1–5 | `7, 1, 5, 4, 3` | không (thêm vào `seen`) | `[]` |
| 6 | `4` | **có** | `[4]` |
| 7–9 | `6, 0, 9` | không | `[4]` |
| 10 | `5` | **có** | `[4, 5]` |
| 11–12 | `8, 2` | không | `[4, 5]` |

Kết quả `[4, 5]` — đúng ví dụ 3.

## 9. Code nộp LeetCode

Bản nộp dùng Cách 3 (tập `seen`): dễ hiểu, `O(n)`, không phụ thuộc giả định về `n`.

In [18]:
class Solution:
    def getSneakyNumbers(self, nums: List[int]) -> List[int]:
        seen = set()                 # các số đã gặp
        ans = []                     # các số bị lặp (đúng 2 phần tử)
        for x in nums:
            if x in seen:            # lần xuất hiện thứ hai
                ans.append(x)
            else:
                seen.add(x)
        return ans


sol = Solution()
for nums, expected in examples:
    print(sol.getSneakyNumbers(nums), "kỳ vọng", expected)

[1, 0] kỳ vọng [0, 1]
[3, 2] kỳ vọng [2, 3]
[4, 5] kỳ vọng [4, 5]


## 10. Vì sao thuật toán đúng

- **Bất biến:** sau khi xử lý `k` phần tử đầu, `seen` đúng là tập các giá trị đã xuất hiện trong `k` phần tử đó.
- **Phát hiện lần hai:** `x in seen` đúng khi `x` đã xuất hiện **trước** vị trí hiện tại, tức đây là lần xuất hiện **thứ hai** (hoặc hơn) của `x`.
- **Không bỏ sót:** đề bảo đảm mỗi số lặp xuất hiện đúng hai lần ⇒ lần thứ hai chắc chắn được gặp và bị bắt.
- **Không thừa:** các số còn lại xuất hiện đúng một lần ⇒ không bao giờ thỏa `x in seen` ⇒ không vào `ans`.
- **Không trùng:** mỗi số lặp chỉ có **một** lần xuất hiện thứ hai ⇒ vào `ans` đúng một lần ⇒ `ans` có đúng hai phần tử khác nhau.
- **Thứ tự bất kỳ** được phép nên thứ tự phát hiện (theo lần xuất hiện thứ hai) là hợp lệ.
- Kiểm chứng thực nghiệm: Mục 12 chạy `5000` mảng ngẫu nhiên đúng ràng buộc, đối chiếu **độc lập** bằng `Counter`.

## 11. Phân tích Time / Space Complexity

Gọi `m = len(nums) = n + 2`.

- **Cách 1 (so từng cặp):** `O(m²)` thời gian, `O(1)` bộ nhớ.
- **Cách 2 (sắp xếp):** `O(m log m)` thời gian; `O(m)` bộ nhớ nếu dùng `sorted`.
- **Cách 3 (`seen`):** `O(m)` thời gian trung bình, `O(m)` bộ nhớ (tối đa `n` phần tử trong `set`).
- **Cách 4 (đếm):** `O(m)` thời gian, `O(n)` bộ nhớ.
- **Cách 5 (toán):** `O(m)` thời gian, **`O(1)`** bộ nhớ.
- Cận dưới thời gian là `O(m)` vì phải đọc hết mảng. Với `n <= 100` mọi cách đều chạy tức thì; sự khác biệt có ý nghĩa khi bài mở rộng ra `n` lớn.

## 12. Bộ test

Đề cho phép **thứ tự bất kỳ**, nên ta không so `==` với danh sách mong đợi một cách máy móc. Ta viết `is_valid_answer(nums, ans)` **độc lập** với mọi cách làm: kết quả phải (1) có đúng `2` phần tử, (2) khác nhau, (3) mỗi phần tử xuất hiện đúng `2` lần trong `nums`, (4) tập kết quả bằng đúng tập các giá trị xuất hiện `2` lần (dùng `Counter`). Với các ví dụ đề ta còn so `sorted(...)`.

In [19]:
def is_valid_answer(nums, ans):
    cnt = Counter(nums)
    twice = {x for x, v in cnt.items() if v == 2}
    return len(ans) == 2 and len(set(ans)) == 2 and set(ans) == twice


sol = Solution()
impls = {
    "bruteforce": sneaky_bruteforce,
    "sort": sneaky_sort,
    "seen": sneaky_seen,
    "array": sneaky_array,
    "math": sneaky_math,
    "sol2": Solution2().getSneakyNumbers,
    "sol3": Solution3().getSneakyNumbers,
    "sol4": Solution4().getSneakyNumbers,
    "sol5": Solution5().getSneakyNumbers,
    "leetcode": sol.getSneakyNumbers,
}


def check_all(nums, expected, label):
    for key, fn in impls.items():
        original = list(nums)
        actual = fn(list(nums))
        assert is_valid_answer(original, actual), (label, key, nums, actual)
        if expected is not None:
            assert sorted(actual) == sorted(expected), (label, key, actual, expected)
    print("PASS", label)


for idx, (nums, expected) in enumerate(examples, 1):
    check_all(nums, expected, f"ví dụ {idx}")

PASS ví dụ 1
PASS ví dụ 2
PASS ví dụ 3


### 12.1. Các trường hợp biên

`n = 2` (nhỏ nhất), `n = 100` (lớn nhất), hai số lặp là hai số nhỏ nhất `0` và `1`, hai số lặp là hai số lớn nhất `n-2` và `n-1`, hai bản sao nằm cạnh nhau, và hai bản sao nằm ở hai đầu mảng (đều tự dựng, hợp lệ theo ràng buộc).

In [20]:
check_all([0, 1, 0, 1], [0, 1], "n = 2: lặp cả hai số")
check_all([1, 0, 1, 0], [0, 1], "n = 2: thứ tự khác")
check_all([0, 0, 1, 1], [0, 1], "n = 2: các bản sao cạnh nhau")
check_all([0, 1, 2, 3, 4, 0, 1], [0, 1], "n = 5: lặp hai số nhỏ nhất")
check_all([0, 1, 2, 3, 4, 3, 4], [3, 4], "n = 5: lặp hai số lớn nhất")
check_all([2, 0, 1, 3, 4, 2, 4], [2, 4], "n = 5: lặp ở hai đầu giá trị, trộn")
big = list(range(100)) + [0, 99]                            # n = 100
check_all(big, [0, 99], "n = 100: lặp 0 và 99, bản sao ở cuối")
big2 = [99, 0] + list(range(100))                           # n = 100, bản sao ở đầu
check_all(big2, [0, 99], "n = 100: bản sao ở đầu")

PASS n = 2: lặp cả hai số
PASS n = 2: thứ tự khác
PASS n = 2: các bản sao cạnh nhau
PASS n = 5: lặp hai số nhỏ nhất
PASS n = 5: lặp hai số lớn nhất
PASS n = 5: lặp ở hai đầu giá trị, trộn
PASS n = 100: lặp 0 và 99, bản sao ở cuối
PASS n = 100: bản sao ở đầu


### 12.2. Phản ví dụ cho các cách làm sai

Ba cách "nghe hợp lý" nhưng sai: (a) trả **mọi số** xuất hiện ít nhất một lần (nhầm ngưỡng), (b) dùng `nums.count(x) > 1` rồi thêm cho **mọi** vị trí (số lặp bị thêm hai lần), (c) `return` ngay khi gặp số lặp đầu tiên. Ta chỉ ra chúng sai ở các ví dụ đề.

In [21]:
def wrong_all_unique(nums):
    return list(set(nums))                                   # trả mọi giá trị khác nhau

def wrong_count_everywhere(nums):
    return [x for x in nums if nums.count(x) > 1]            # mỗi số lặp xuất hiện 2 lần -> thêm 2 lần

def wrong_return_early(nums):
    seen = set()
    for x in nums:
        if x in seen:
            return [x]                                       # chỉ có một số
        seen.add(x)


for name, wrong in [("mọi giá trị", wrong_all_unique), ("count khắp nơi", wrong_count_everywhere), ("return sớm", wrong_return_early)]:
    nums = [0, 3, 2, 1, 3, 2]
    got = wrong(nums)
    print(f"cách '{name}': trả {got} -> hợp lệ? {is_valid_answer(nums, got)}")
    assert not is_valid_answer(nums, got)

cách 'mọi giá trị': trả [0, 1, 2, 3] -> hợp lệ? False
cách 'count khắp nơi': trả [3, 2, 3, 2] -> hợp lệ? False
cách 'return sớm': trả [3] -> hợp lệ? False


### 12.3. Đối chiếu ngẫu nhiên (có seed) đúng ràng buộc

Sinh `5000` mảng hợp lệ: chọn `n` trong `[2, 100]`, lấy `0..n-1`, chọn **hai số khác nhau** để lặp thêm, rồi xáo trộn (`random.Random(3289)`). Kiểm tra mảng sinh ra đúng ràng buộc rồi đối chiếu **mọi cách** bằng `is_valid_answer`; ngoài ra chúng phải cho **cùng tập** kết quả. Thêm một vét cạn nhỏ: với `n = 4`, thử **mọi** cặp số lặp và **mọi** cách xáo trộn khác nhau của mảng dài `6` (lấy bằng `itertools.permutations`, khử trùng bằng `set`).

In [22]:
import itertools

rng = random.Random(3289)
for trial in range(5000):
    n = rng.choice([2, 3, 4, 5, 10, 50, 99, 100]) if trial % 2 == 0 else rng.randint(2, 100)
    a, b = rng.sample(range(n), 2)
    nums = list(range(n)) + [a, b]
    rng.shuffle(nums)
    assert len(nums) == n + 2 and all(0 <= x < n for x in nums)
    assert sorted(Counter(nums).values()).count(2) == 2          # đúng hai số xuất hiện 2 lần
    ref = sorted([a, b])
    for key, fn in impls.items():
        got = fn(list(nums))
        assert is_valid_answer(nums, got), (key, nums, got)
        assert sorted(got) == ref, (key, nums, got)
print("PASS 5000 mảng ngẫu nhiên đúng ràng buộc")

count = 0
n = 4
for a, b in itertools.combinations(range(n), 2):
    base = list(range(n)) + [a, b]
    for perm in set(itertools.permutations(base)):
        nums = list(perm)
        for key, fn in impls.items():
            assert sorted(fn(list(nums))) == [a, b], (key, nums)
        count += 1
print("PASS vét cạn n = 4:", count, "mảng (mọi cặp lặp, mọi cách xáo trộn)")

PASS 5000 mảng ngẫu nhiên đúng ràng buộc
PASS vét cạn n = 4: 1080 mảng (mọi cặp lặp, mọi cách xáo trộn)


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Dùng `count` rồi thêm mọi lần xuất hiện (kết quả thừa)

In [23]:
def sai_count(nums):
    ans = []
    for x in nums:
        if nums.count(x) > 1:        # cả hai bản sao của x đều thỏa -> x vào ans 2 lần
            ans.append(x)
    return ans

print(sai_count([0, 3, 2, 1, 3, 2]), "(đúng là 2 số: [3, 2])")     # [3, 2, 3, 2]

[3, 2, 3, 2] (đúng là 2 số: [3, 2])


**Vì sao sai:** mỗi số lặp có **hai bản sao**, và **cả hai** đều thỏa `count > 1`, nên số đó vào kết quả hai lần (kết quả dài `4`). Ngoài ra `count` trong vòng lặp tốn `O(n²)`. Cách đúng: chỉ ghi nhận ở **lần xuất hiện thứ hai** (`x in seen`) hoặc dùng `set(...)` để khử trùng.

### Lỗi 2: `IndexError` khi so phần tử kề nhau sau khi sắp xếp

In [24]:
def sai_index(nums):
    b = sorted(nums)
    ans = []
    for i in range(len(b)):              # i đi tới len - 1 -> b[i + 1] vượt biên
        if b[i] == b[i + 1]:
            ans.append(b[i])
    return ans

try:
    sai_index([0, 3, 2, 1, 3, 2])
except IndexError as e:
    print("IndexError:", e)

IndexError: list index out of range


**Vì sao sai:** ở `i = len(b) - 1` ta truy cập `b[i + 1]` — **ngoài mảng**. Vì có `b[i + 1]`, vòng lặp phải là `range(len(b) - 1)`.

### Lỗi 3: `return` sớm — chỉ tìm được một số

In [25]:
def sai_return_som(nums):
    seen = set()
    for x in nums:
        if x in seen:
            return [x]                  # dừng quá sớm, còn một số lặp nữa
        seen.add(x)

print(sai_return_som([0, 3, 2, 1, 3, 2]), "(thiếu 2)")      # [3]

[3] (thiếu 2)


**Vì sao sai:** đề có **hai** số lặp; `return` ngay lần đầu bỏ lỡ số còn lại. Phải duyệt **hết** mảng rồi mới trả `ans`.

### Lỗi 4: Dùng `/` thay `//` (ra số thực) trong cách toán học

In [26]:
def sai_chia_thuc(nums):
    n = len(nums) - 2
    s = sum(nums) - n * (n - 1) / 2
    q = sum(x * x for x in nums) - (n - 1) * n * (2 * n - 1) / 6
    d = math.isqrt(int(2 * q - s * s))
    return [(s + d) / 2, (s - d) / 2]

print(sai_chia_thuc([0, 3, 2, 1, 3, 2]), "(đúng là số nguyên [3, 2])")      # [3.0, 2.0]

[3.0, 2.0] (đúng là số nguyên [3, 2])


**Vì sao sai:** `/` luôn cho **số thực** (`3.0`), còn đề cần **số nguyên** (`3`). Với `n` lớn hơn, số thực còn có thể sai số làm tròn. Hãy dùng `//` và `math.isqrt` để mọi thứ giữ là số nguyên.

**Code đúng (nhắc lại):** tập `seen`, `if x in seen: ans.append(x)` rồi `seen.add(x)`.

In [27]:
def dung(nums):
    seen = set()
    ans = []
    for x in nums:
        if x in seen:
            ans.append(x)
        else:
            seen.add(x)
    return ans

for nums, expected in examples:
    got = dung(nums)
    assert is_valid_answer(nums, got) and sorted(got) == sorted(expected)
print("PASS code đúng")

PASS code đúng


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "**Tìm số trùng/lặp**" ⇒ `set` (đã gặp chưa?) hoặc bộ đếm (`Counter`). Ghi nhớ: **kiểm tra `in` trước, rồi mới thêm**.
- Giá trị nằm trong `0..n-1` ⇒ dùng **mảng đếm** thay cho `dict`, chỉ số **là** giá trị.
- "**Mỗi phần tử đúng một lần, trừ vài phần tử**" ⇒ cân nhắc toán học (tổng, tổng bình phương) hoặc XOR để dùng `O(1)` bộ nhớ.
- Không muốn thêm bộ nhớ ⇒ **sắp xếp** rồi so phần tử kề nhau (nhưng nhớ `sorted` vs `.sort()`).
- "**Thứ tự bất kỳ**" ⇒ khi tự test, so `sorted(...)` hoặc dùng hàm kiểm tra tính hợp lệ độc lập.
- Cẩn thận **ngưỡng tần suất**: đề nói "đúng hai lần" ⇒ `== 2`; nếu có thể nhiều hơn thì dùng `>= 2` và khử trùng.

## 15. Biến thể của bài

1. **Tần suất tổng quát:** giá trị xuất hiện **từ hai lần trở lên** (dùng `>= 2`, trả danh sách không trùng).
2. **Chỉ một số lặp:** mảng `0..n-1` thêm đúng một số lặp (LeetCode 287 dạng dễ).
3. **Thứ tự theo lần xuất hiện thứ hai** hoặc **theo thứ tự tăng dần** (`sorted`).
4. **Trả vị trí** (chỉ số) của hai bản sao thay vì giá trị.
5. **XOR:** dùng phép `^` để tìm hai số lặp với `O(1)` bộ nhớ (xem Bài 5(b)).

In [28]:
def repeated_values(nums):
    """Mọi giá trị xuất hiện >= 2 lần, không trùng, tăng dần."""
    return sorted(x for x, v in Counter(nums).items() if v >= 2)


def one_sneaky(nums):
    """Mảng chứa 0..n-1 và đúng một số lặp thêm: trả số đó."""
    seen = set()
    for x in nums:
        if x in seen:
            return x
        seen.add(x)


def sneaky_with_positions(nums):
    """{giá trị lặp: [vị trí 1, vị trí 2]}"""
    pos = {}
    for i, x in enumerate(nums):
        pos.setdefault(x, []).append(i)
    return {x: p for x, p in pos.items() if len(p) == 2}


print(repeated_values([0, 3, 2, 1, 3, 2, 3]))            # [2, 3]
print(one_sneaky([0, 2, 1, 3, 2]))                       # 2
print(sneaky_with_positions([0, 3, 2, 1, 3, 2]))         # {3: [1, 4], 2: [2, 5]}
print(sorted(Solution().getSneakyNumbers([0, 3, 2, 1, 3, 2])))   # [2, 3]

[2, 3]
2
{3: [1, 4], 2: [2, 5]}
[2, 3]


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. So từng cặp | hai vòng `for`, `nums[i] == nums[j]` | `O(n²)` | `O(1)` | `n` rất nhỏ, bám sát định nghĩa |
| 2. Sắp xếp | `sorted` rồi so `b[i]` với `b[i+1]` | `O(n log n)` | `O(n)` | không muốn dùng hash, hoặc cần kết quả tăng dần |
| 3. Tập `seen` | `x in seen` ⇒ lần thứ hai | `O(n)` | `O(n)` | **nên nộp**, tổng quát và rõ ràng |
| 4. Mảng đếm / `Counter` | đếm, lấy giá trị đếm `== 2` | `O(n)` | `O(n)` | khi cần cả bảng tần suất |
| 5. Toán `S`, `Q` | `a+b`, `a²+b²` rồi giải | `O(n)` | `O(1)` | cần `O(1)` bộ nhớ, đề bảo đảm đúng 2 số lặp |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `3 in {1, 2, 3}`, `[1, 1, 2].count(1)`, `sorted([3, 1, 2])`, `sorted(Solution().getSneakyNumbers([0, 1, 1, 0]))`, và `n` khi `len(nums) = 12`.

**Bài 2 (tính tay).** Với `nums = [1, 4, 2, 0, 3, 4, 2]` (`n = 5`): lập bảng từng bước của tập `seen` và cho biết hai số lặp theo thứ tự phát hiện.

**Bài 3 (điền TODO).** Hoàn thành `sneaky_todo(nums)` theo Cách 3.

In [29]:
def sneaky_todo(nums):
    # TODO 1: tạo seen = set() và ans = []
    # TODO 2: duyệt từng x trong nums
    #         nếu x đã có trong seen -> ans.append(x); ngược lại -> seen.add(x)
    # TODO 3: trả về ans
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `sneaky_sort2(nums)` theo Cách 2 nhưng bằng `zip(b, b[1:])` (duyệt từng cặp kề nhau) thay cho chỉ số `i`, `i + 1`.

**Bài 5 (biến thể / mini challenge).** (a) Viết `repeated_any(nums)` trả danh sách **tăng dần, không trùng** các giá trị xuất hiện **ít nhất hai lần** (đầu vào bất kỳ, không cần theo ràng buộc đề). (b) Viết `sneaky_xor(nums)` tìm hai số lặp bằng **XOR** với `O(1)` bộ nhớ.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `True`, `2`, `[1, 2, 3]`, `[0, 1]`, `10` (`n = len(nums) - 2`). Bài 3: `if x in seen: ans.append(x) else: seen.add(x)`.
- **Mức 2 (vừa):** Bài 2: gặp `1, 4, 2, 0, 3` đều chưa có (`seen = {0, 1, 2, 3, 4}`), rồi `4` đã có (lặp), `2` đã có (lặp) ⇒ `[4, 2]`. Bài 4: `[a for a, b2 in zip(b, b[1:]) if a == b2]`.
- **Mức 3 (sâu):** Bài 5(a): `Counter` rồi lọc `v >= 2` và `sorted`. 5(b): XOR mọi phần tử của `nums` với XOR của `0..n-1` được `x = a ^ b` (các số còn lại triệt tiêu). `x != 0` nên có bit `1`: lấy `low = x & -x`. Chia mọi giá trị (cả `nums` lẫn `0..n-1`) thành hai nhóm theo `giá_trị & low`; XOR từng nhóm cho ra `a` và `b` (hai số rơi vào hai nhóm khác nhau; các số khác xuất hiện chẵn lần mỗi nhóm nên triệt tiêu).

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [30]:
# Bài 1
assert (3 in {1, 2, 3}) is True
assert [1, 1, 2].count(1) == 2
assert sorted([3, 1, 2]) == [1, 2, 3]
assert sorted(Solution().getSneakyNumbers([0, 1, 1, 0])) == [0, 1]
assert 12 - 2 == 10

# Bài 2
assert Solution().getSneakyNumbers([1, 4, 2, 0, 3, 4, 2]) == [4, 2]


# Bài 3
def sneaky_todo_ans(nums):
    seen = set()
    ans = []
    for x in nums:
        if x in seen:
            ans.append(x)
        else:
            seen.add(x)
    return ans


for nums, expected in examples:
    assert sorted(sneaky_todo_ans(nums)) == sorted(expected)


# Bài 4
def sneaky_sort2(nums):
    b = sorted(nums)
    return [a for a, c in zip(b, b[1:]) if a == c]


for nums, expected in examples:
    assert sorted(sneaky_sort2(nums)) == sorted(expected)


# Bài 5(a)
def repeated_any(nums):
    return sorted(x for x, v in Counter(nums).items() if v >= 2)


assert repeated_any([5, 1, 5, 2, 2, 2, 9]) == [2, 5]
assert repeated_any([1, 2, 3]) == []
assert repeated_any([]) == []


# Bài 5(b)
def sneaky_xor(nums):
    n = len(nums) - 2
    x = 0
    for v in nums:
        x ^= v
    for v in range(n):
        x ^= v                          # x = a ^ b
    low = x & -x                        # một bit khác nhau giữa a và b
    a = 0
    for v in nums:
        if v & low:
            a ^= v
    for v in range(n):
        if v & low:
            a ^= v                      # a = một trong hai số lặp
    return [a, x ^ a]


for nums, expected in examples:
    assert sorted(sneaky_xor(nums)) == sorted(expected)
rng2 = random.Random(32890)
for _ in range(2000):
    n = rng2.randint(2, 100)
    a, b = rng2.sample(range(n), 2)
    nums = list(range(n)) + [a, b]
    rng2.shuffle(nums)
    assert sorted(sneaky_xor(nums)) == sorted([a, b])
print("PASS toàn bộ đáp án")

PASS toàn bộ đáp án


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (tập `seen`: gặp lần hai thì ghi nhận)

```python
class Solution:
    def getSneakyNumbers(self, nums: List[int]) -> List[int]:
        seen = set()
        ans = []
        for x in nums:
            if x in seen:
                ans.append(x)
            else:
                seen.add(x)
        return ans
```

**Checklist**

- [ ] Kiểm tra `x in seen` **trước** khi thêm `x` vào `seen`?
- [ ] Duyệt **hết** mảng (không `return` sớm)? Kết quả có đúng `2` phần tử khác nhau?
- [ ] Mỗi số lặp chỉ được ghi **một lần** (ở lần xuất hiện thứ hai)?
- [ ] Nếu sắp xếp: dùng `sorted` (hoặc chấp nhận sửa mảng), vòng lặp `range(len - 1)` để tránh `IndexError`?
- [ ] Nếu dùng toán: `//` và `math.isqrt`, kết quả là số nguyên?
- [ ] Đã thử ba ví dụ đề, `n = 2`, `n = 100`, và bản sao ở hai đầu mảng?
- [ ] Skeleton đúng `class Solution` / `def getSneakyNumbers(self, nums: List[int]) -> List[int]`?